# Stage 1.5 -- Validation & Feature Engineering: Panel C (Macro Daily)

## Input
`Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_daily.parquet` (142 columns, ~5,275 rows, keyed on `date` only)

## Purpose
Validation and feature engineering of the merged macro daily panel. Unlike Panels A and B (stock-level), Panel C is market-level data -- every factor is a single daily time series spanning 2004--2024. There is no cross-sectional dimension and no PERMNO. Raw levels (FX rates, CFTC contract counts, yields) are appropriate here because they capture regime changes; z-scoring in the model pipeline handles any trends. No normalisation by market cap is needed.

---

## Initial Diagnostics

A diagnostic pass lists all factor columns grouped by theme (Treasury yields, TIPS/breakevens, policy rates, credit spreads, commodities, FX, VIX family, Fama-French, world indices, VIX futures, SKEW, CFTC positioning, AAII sentiment, FRED weekly) with NaN rates, min/max ranges, and identifies non-numeric columns and exact duplicates.

---

## Block 1: Clean & Diagnose

### Step 1: Replace Infinite Values with NaN
Safety net scan across all numeric columns. No infinite values found -- panel is already clean.

### Step 2: Near-Zero Variance Check
Scans all factor columns for effectively constant values (std < 1e-8). No near-zero variance factors found.

### Step 3: Structural NaN Patterns
Reports all columns with any NaN, with counts, percentages, and first valid dates. Expected structural patterns confirmed:
- CFTC positioning (~11.7% NaN): data starts June 2006 (disaggregated report began then)
- Trade-weighted dollar indices `twexb`/`twexm` (~9.5% NaN): series starts January 2006
- VIX futures (~1.1% NaN): launched March 26, 2004
- SKEW warmup periods (~2.5% NaN for `skew_pctile_252d`): 252-day rolling lookback

### Step 4: Drop Assessment
No factors dropped. All 141 are meaningful time series:
- `rf` retained despite only 3 unique values (0.0000 during ZIRP, 0.0001--0.0002 otherwise) because it captures monetary policy stance and is needed for excess return computation
- `twexb`/`twexm` retained despite 9.5% NaN (structural late start)
- CFTC retained despite 11.7% NaN (structural late start)

### Gap Analysis (Post-June 2006)
A detailed investigation of the NaN landscape after the latest-starting structural series (CFTC, June 2006):
- **Pre vs post June 2006 NaN comparison:** columns with any post-2006 NaN are listed with rates in both periods
- **Largest consecutive NaN gaps post-2006:** top 20 gaps identified with start/end dates (all are short, 1--3 day foreign holiday gaps in world index returns and VXN)
- **Forward-fill extent for weekly sources:** confirms merge_asof from Stage 1 correctly carries weekly values forward (~5 trading days persistence for CFTC, AAII, FRED weekly)
- **CFTC update frequency verification:** confirms ~5 trading day persistence between weekly updates

### Post-June 2006 Cleanup
- Data trimmed to post-June 2006 for a clean, complete panel
- 7 sporadic single-day foreign holiday gaps in `vxn`, `vxno`, `vxnh`, `vxnl`, `widx_chn`, `widx_jpn`, `widx_kor` filled with `ffill(limit=2)`
- Verified zero NaN remaining after cleanup

---

## Block 2: Complete Factor Inventory (Pre-Engineering)

Every surviving factor catalogued with column name, source, category, and description. Organised into themed groups: Treasury Yields (11), TIPS & Breakevens (6), Policy Rates (4), Credit Spreads OAS (9), Moody's Yields (2), Yield Curve Derived (13), Commodities (4), FX Trade-Weighted (2), FX Rates (11), VIX Family (12), Fama-French + Momentum (6), World Index Returns (12), VIX Futures & Term Structure (9), SKEW (6), CFTC Positioning (22), AAII Sentiment (5), FRED Weekly (7).

The inventory is validated bidirectionally against the actual data columns and saved as `macro_daily_descriptions_pre.csv`.

---

## Block 3: Feature Engineering

Six sections creating ~90 new features and dropping 11 redundant/replaced columns.

### Section A: Drop Redundancies & Derive VIX Intraday Features (8 new, 11 dropped)

**Exact duplicates dropped (2):** `real_rate_10y` confirmed identical to `tips_10y`, `real_rate_5y` confirmed identical to `tips_5y` (TIPS yield IS the real rate).

**VIX family intraday features derived (6):** for each of VIX, VXN, VXD:
- `{prefix}_intraday_range` = (high - low) / close
- `{prefix}_overnight_gap` = (open - prev close) / prev close

**VIX OHLC columns dropped (9):** `vixo`, `vixh`, `vixl`, `vxno`, `vxnh`, `vxnl`, `vxdo`, `vxdh`, `vxdl` -- information now captured by range and gap features. Only VIX/VXN/VXD close values retained.

**VIX cross-index ratios (2):** `vix_vxn_ratio` = VIX / VXN (S&P vol relative to NASDAQ vol), `vix_vxd_ratio` = VIX / VXD (S&P vol relative to DJIA vol).

### Section B: Daily Changes in Key Levels -- 1d and 5d (21 features)

Raw differences (not percentage changes) for rates and spreads, since these move in basis points:
- **Yield changes:** 2Y, 10Y, 30Y (1d and 5d each)
- **Yield curve slope changes:** 2Y10Y, 3M10Y (1d and 5d each)
- **Credit spread changes:** HY OAS, IG OAS, BBB-AAA spread (1d and 5d each)
- **VIX changes:** absolute 1d, absolute 5d, percentage 1d
- **Breakeven inflation changes:** 10Y breakeven (1d and 5d)

### Section C: Returns from Level Data -- FX, Commodities, Dollar Index (13 features)

Percentage changes for asset prices where returns (not absolute changes) are the economically meaningful quantity:
- **FX daily returns (5):** EUR, JPY, GBP, AUD, CNY (key pairs only -- model has 11 FX levels for regime)
- **Dollar index returns (2):** broad and major trade-weighted
- **Commodity returns (6):** WTI oil, Brent oil, natural gas (1d and 5d each)

### Section D: Rolling Dynamics (20 features)

**D1. Moving average deviations (4):** VIX vs 20d/50d MA (percentage deviation), 10Y yield vs 20d MA (basis point deviation), HY OAS vs 20d MA.

**D2. Cumulative factor returns (8):** 5d and 20d cumulative sums for `mktrf`, `smb`, `hml`, `umd` (factor momentum).

**D3. Volatility of key series (5):** market return 5d/20d rolling std + vol ratio (5d/20d), yield change 20d rolling std, HY OAS change 20d rolling std.

**D4. Global equity momentum (3):** `widx_avg_ret` (cross-country average world index return), `widx_avg_cum_5d` (5d cumulative), `widx_dispersion` (cross-country return standard deviation -- global divergence measure).

### Section E: Regime & Cross-Asset Signals (7 features)

- **VIX regime indicators (2):** `vix_above_20` (elevated fear), `vix_above_30` (crisis)
- **Yield curve inversion indicators (2):** `curve_inverted_2y10y`, `curve_inverted_3m10y`
- **Credit stress indicator (1):** `credit_stress` = HY OAS > 5%
- **Stock-bond correlation (1):** `stock_bond_corr_20d` = 20-day rolling correlation between market return and yield change
- **Risk appetite composite (1):** `risk_appetite` = average of z-scored(mktrf) + z-scored(-delta HY OAS) + z-scored(-delta VIX), all using 50-day expanding z-scores. Positive = risk-on environment.

### Section F: Calendar Features (8 features)

All using the TRADING calendar (actual rows in the DataFrame), not calendar days:
- `day_of_week` (Mon=0, Fri=4), `is_monday`, `is_friday` -- well-documented day-of-week effects
- `month_of_year` (Jan=1, Dec=12) -- January effect, tax-loss selling
- `is_quarter_end` -- rebalancing, window dressing
- `trading_days_to_month_end` -- counts actual trading rows remaining in month, not calendar days
- `is_turn_of_month` -- last 2 + first 2 trading days of month (institutional rebalancing flows). Fixed post-Block 3 to use exact trading day rank within month rather than calendar day approximation.
- `is_opex_week` -- options expiration week (week containing the 3rd Friday). Generated mathematically; survives Good Friday (when 3rd Friday is a market holiday, OPEX moves to Thursday but the full week is still flagged).

Calendar features verified: `is_monday` mean ~0.20, `is_friday` mean ~0.20, `is_opex_week` mean ~0.20, `is_turn_of_month` mean ~0.19, `trading_days_to_month_end` range [0, ~22].

---

## Block 4: Final Factor Inventory & Save

Every surviving factor catalogued in a final inventory DataFrame with column name, source, category, and description. Validated

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PANEL_C_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_daily.parquet')

df = pd.read_parquet(PANEL_C_PATH)
df['date'] = pd.to_datetime(df['date'])

factor_cols = [c for c in df.columns if c != 'date']

print(f"Panel C: {len(df):,} rows × {df.shape[1]} columns")
print(f"Factors: {len(factor_cols)}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# Non-numeric check
non_numeric = [c for c in factor_cols if not pd.api.types.is_numeric_dtype(df[c])]
print(f"Non-numeric: {non_numeric}")

# Infinite check
print(f"\nInfinite values:")
for c in factor_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        n_inf = np.isinf(df[c]).sum()
        if n_inf > 0:
            print(f"  {c}: {n_inf}")

# Full column list with NaN and range
print(f"\n{'#':<5} {'Column':<35} {'NaN%':>7}  {'Min':>14}  {'Max':>14}")
print("-" * 80)
for i, c in enumerate(factor_cols, 1):
    nan_p = df[c].isna().mean() * 100
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 0:
            try:
                print(f"{i:<5} {c:<35} {nan_p:>6.2f}%  {float(vals.min()):>14.4f}  {float(vals.max()):>14.4f}")
            except:
                print(f"{i:<5} {c:<35} {nan_p:>6.2f}%  {'error':>14}  {'error':>14}")
        else:
            print(f"{i:<5} {c:<35} {nan_p:>6.2f}%  {'all NaN':>14}  {'':>14}")
    else:
        print(f"{i:<5} {c:<35} {nan_p:>6.2f}%  {'non-numeric':>14}  {'':>14}")

# Quick correlation check for exact duplicates
print(f"\n--- Exact duplicate check (sample 2000 rows) ---")
numeric_factors = [c for c in factor_cols if pd.api.types.is_numeric_dtype(df[c])]
sample = df[numeric_factors].sample(min(2000, len(df)), random_state=42)
exact_dupes = []
for i, c1 in enumerate(numeric_factors):
    for c2 in numeric_factors[i+1:]:
        s1 = sample[c1]
        s2 = sample[c2]
        if s1.isna().equals(s2.isna()):
            valid = pd.DataFrame({'a': s1, 'b': s2}).dropna()
            if len(valid) > 100 and valid['a'].equals(valid['b']):
                exact_dupes.append((c1, c2))
if exact_dupes:
    print(f"  Found {len(exact_dupes)} exact duplicate pairs:")
    for c1, c2 in exact_dupes:
        print(f"    {c1} == {c2}")
else:
    print(f"  ✓ No exact duplicates")

Panel C: 5,285 rows × 142 columns
Factors: 141
Date range: 2004-01-02 → 2024-12-31
Non-numeric: []

Infinite values:

#     Column                                 NaN%             Min             Max
--------------------------------------------------------------------------------
1     yield_1m                              0.00%          0.0000          6.0200
2     yield_3m                              0.00%         -0.0500          5.3600
3     yield_6m                              0.00%          0.0200          5.3600
4     yield_1y                              0.00%          0.0400          5.4900
5     yield_2y                              0.00%          0.0900          5.2900
6     yield_3y                              0.00%          0.1000          5.2600
7     yield_5y                              0.00%          0.1900          5.2300
8     yield_7y                              0.00%          0.3600          5.2300
9     yield_10y                             0.00%          0.52

In [2]:
# %% [markdown]
# # Stage 1.5 — Validation & Feature Engineering: Panel C (Macro Daily)
#
# Panel C is market-level data keyed on (date) only — no permno, no
# cross-sectional aggregation needed. Every factor is a single daily
# time series spanning 2004-2024.
#
# Block 1: Replace infinites, diagnostics, drop truly redundant factors
# Block 2: Inventory of surviving factors
# Block 3: Feature engineering (yield curve, FX returns, regime indicators, etc.)
# Block 4: Final inventory & save
#
# Input:  Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_daily.parquet
# Output: Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

IN_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_Initial_Merge/panel_macro_daily.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(IN_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
n_start = df.shape[1]

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 1: CLEAN & DIAGNOSE
# ═══════════════════════════════════════════════════════════════════════════════

# %% [markdown]
# ## Block 1: Clean & Diagnose
#
# Panel C is already very clean — no non-numeric columns, no infinites,
# no exact duplicates. This block:
#
# 1. Replaces ±inf with NaN (safety net for any edge cases)
# 2. Runs near-zero variance check
# 3. Checks structural NaN patterns (CFTC starts Jun 2006, VIX futures Mar 2004)
# 4. Identifies any drops — expected to be minimal
#
# Key insight: Panel C factors are TIME SERIES, not cross-sectional.
# Raw levels (FX rates, CFTC contract counts, yields) are appropriate
# here — they capture regime changes. Z-scoring in Step 3 handles
# any trends. No normalisation by market cap is needed.

# %%
print("=" * 90)
print("BLOCK 1: CLEAN & DIAGNOSE")
print("=" * 90)

# ── Step 1: Replace ±inf with NaN ────────────────────────────────────────────
print("\n--- Step 1: Replace ±inf with NaN ---")

numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_count = 0
for c in numeric_cols:
    n_inf = np.isinf(df[c]).sum()
    if n_inf > 0:
        print(f"  {c}: {n_inf} infinite values")
        inf_count += n_inf

df = df.replace([np.inf, -np.inf], np.nan)

if inf_count == 0:
    print(f"  ✓ No infinite values found (clean)")
else:
    print(f"  Replaced {inf_count} infinite values with NaN")

# ── Step 2: Near-zero variance check ────────────────────────────────────────
print("\n--- Step 2: Near-zero variance check ---")

factor_cols = [c for c in df.columns if c != 'date']
low_var_found = []
for c in factor_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        vals = df[c].dropna()
        if len(vals) > 100:
            std = vals.std()
            if pd.notna(std) and float(std) < 1e-8:
                low_var_found.append((c, vals.nunique(), float(std), float(vals.min()), float(vals.max())))
                print(f"  {c:<35s} nunique={vals.nunique():>6d}  std={float(std):.2e}  "
                      f"range=[{float(vals.min()):.6f}, {float(vals.max()):.6f}]")

if not low_var_found:
    print(f"  ✓ No near-zero variance factors")

# ── Step 3: Structural NaN patterns ─────────────────────────────────────────
print("\n--- Step 3: Structural NaN patterns ---")
print("  (Expected: CFTC starts Jun 2006, VIX futures Mar 2004, twex late-starting)\n")

nan_by_col = df[factor_cols].isna().sum()
nan_cols = nan_by_col[nan_by_col > 0].sort_values(ascending=False)

if len(nan_cols) > 0:
    print(f"  {'Column':<35s} {'NaN':>6s}  {'%':>6s}  {'First Valid':>12s}")
    print("  " + "-" * 65)
    for col in nan_cols.index:
        n = int(nan_cols[col])
        pct = n / len(df) * 100
        first_valid = df[df[col].notna()]['date'].min()
        fv_str = first_valid.strftime('%Y-%m-%d') if pd.notna(first_valid) else 'never'
        print(f"  {col:<35s} {n:>6,d}  {pct:>5.2f}%  {fv_str:>12s}")
else:
    print(f"  ✓ No NaN in any column")

# ── Step 4: Check for factors that should be dropped ─────────────────────────
print("\n--- Step 4: Drop assessment ---")

# Panel C is time-series data — levels are appropriate (z-scored in Step 3).
# No cross-sectional normalisation needed.
# No price-level columns to transform (everything is market-level).
#
# Potential drops:
# - rf (daily risk-free rate): range [0.0000, 0.0002]. Essentially zero during
#   ZIRP years. BUT the level of rf is meaningful — it's the same as
#   fed_funds_eff/252 and captures monetary policy stance. KEEP IT.
# - twexb/twexm at 9.54% NaN: late-starting trade-weighted dollar indices.
#   These have structural NaN at the start (series didn't exist). KEEP —
#   the model can handle NaN in the first ~500 days.

# Nothing to drop in Panel C — all factors are meaningful time series.
drop_cols = []

if drop_cols:
    drops_present = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=drops_present)
    print(f"\n  Dropped {len(drops_present)} columns: {drops_present}")
else:
    print(f"\n  ✓ No factors dropped — all 141 are meaningful time series")
    print(f"    rf: near-zero during ZIRP but captures monetary policy stance")
    print(f"    twexb/twexm: 9.5% NaN is structural (late-starting series)")
    print(f"    CFTC 11.7% NaN: structural (data starts Jun 2006)")
    print(f"    VIX futures 1.1% NaN: structural (launches Mar 2004)")

# ── Step 5: Summary ─────────────────────────────────────────────────────────
print(f"\n  Columns: {n_start} → {df.shape[1]} ({n_start - df.shape[1]} dropped)")

remaining = [c for c in df.columns if c != 'date']
print(f"  Remaining factors: {len(remaining)}")

# Verify date column intact
assert 'date' in df.columns, "FATAL: date column missing!"
assert df['date'].is_monotonic_increasing, "FATAL: dates not sorted!"
print(f"  ✓ Date column intact and sorted")

# ═══════════════════════════════════════════════════════════════════════════════
# POST-BLOCK 1: DIAGNOSTIC OUTPUT FOR BLOCK 2
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("SURVIVING COLUMNS FOR BLOCK 2")
print("=" * 90)

# Group columns by source/theme for easier review
groups = {
    'Treasury Yields': [c for c in remaining if c.startswith('yield_')],
    'TIPS & Breakevens': [c for c in remaining if c.startswith(('tips_', 'breakeven_'))],
    'Policy Rates': [c for c in remaining if c in ['fed_funds_eff', 'prime_rate', 'discount_rate', 'rf']],
    'Credit Spreads (OAS)': [c for c in remaining if c.endswith('_oas')],
    'Moody\'s Yields': [c for c in remaining if c.startswith('moody_')],
    'Yield Curve Derived': [c for c in remaining if c.startswith(('slope_', 'curve_', 'real_rate_', 'ff_'))
                            or c in ['bbb_aaa_spread', 'bb_bbb_spread', 'moody_baa_aaa']],
    'Commodities': [c for c in remaining if c in ['wti_oil', 'brent_oil', 'natgas', 'brent_wti_spread']],
    'FX Trade-Weighted': [c for c in remaining if c.startswith('twex')],
    'FX Rates': [c for c in remaining if c.startswith('fx_')],
    'VIX Family': [c for c in remaining if c in ['vix', 'vixo', 'vixh', 'vixl',
                   'vxn', 'vxno', 'vxnh', 'vxnl', 'vxd', 'vxdo', 'vxdh', 'vxdl']],
    'Fama-French': [c for c in remaining if c in ['mktrf', 'smb', 'hml', 'rmw', 'cma', 'umd']],
    'World Index Returns': [c for c in remaining if c.startswith('widx_')],
    'VIX Futures & Term': [c for c in remaining if c.startswith('vix_fut_') or c.startswith('vix_term_')
                           or c == 'vix_futures_basis'],
    'SKEW': [c for c in remaining if c.startswith('skew')],
    'CFTC Positioning': [c for c in remaining if c in [
        'lev_long', 'lev_short', 'lev_spread', 'am_long', 'am_short', 'am_spread',
        'dealer_long', 'dealer_short', 'dealer_spread', 'other_long', 'other_short',
        'other_spread', 'open_interest', 'lev_net', 'am_net', 'dealer_net',
        'lev_net_pct', 'am_net_pct', 'dealer_net_pct', 'lev_am_ratio',
        'lev_net_chg', 'am_net_chg']],
    'AAII Sentiment': [c for c in remaining if c in ['bullish', 'neutral', 'bearish',
                       'bullish_8w_ma', 'bull_bear_spread']],
    'FRED Weekly': [c for c in remaining if c in ['initial_claims', 'continued_claims',
                    'fed_assets', 'tga', 'reserves', 'bank_credit', 'ci_loans']],
}

# Catch any ungrouped columns
grouped = set()
for cols in groups.values():
    grouped.update(cols)
ungrouped = [c for c in remaining if c not in grouped]

print(f"\n  Total surviving factors: {len(remaining)}")
for name, cols in groups.items():
    if cols:
        print(f"\n  {name} ({len(cols)}):")
        for c in cols:
            nan_p = df[c].isna().mean() * 100
            vals = df[c].dropna()
            mn = f"{float(vals.min()):.4f}" if len(vals) > 0 else "N/A"
            mx = f"{float(vals.max()):.4f}" if len(vals) > 0 else "N/A"
            print(f"    {c:<35s} nan={nan_p:>5.2f}%  range=[{mn}, {mx}]")

if ungrouped:
    print(f"\n  ⚠ UNGROUPED ({len(ungrouped)}):")
    for c in ungrouped:
        nan_p = df[c].isna().mean() * 100
        print(f"    {c:<35s} nan={nan_p:>5.2f}%")

Loaded: 5,285 rows × 142 columns
BLOCK 1: CLEAN & DIAGNOSE

--- Step 1: Replace ±inf with NaN ---
  ✓ No infinite values found (clean)

--- Step 2: Near-zero variance check ---
  ✓ No near-zero variance factors

--- Step 3: Structural NaN patterns ---
  (Expected: CFTC starts Jun 2006, VIX futures Mar 2004, twex late-starting)

  Column                                 NaN       %   First Valid
  -----------------------------------------------------------------
  am_net_chg                             624  11.81%    2006-06-26
  lev_net_chg                            624  11.81%    2006-06-26
  lev_spread                             619  11.71%    2006-06-19
  open_interest                          619  11.71%    2006-06-19
  am_long                                619  11.71%    2006-06-19
  am_short                               619  11.71%    2006-06-19
  am_spread                              619  11.71%    2006-06-19
  dealer_long                            619  11.71%    2006-06-19

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# GAP ANALYSIS: What does the NaN landscape look like post-2006?
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

factor_cols = [c for c in df.columns if c != 'date']

# ── 1. NaN rates BEFORE vs AFTER June 2006 ──────────────────────────────────
cutoff = pd.Timestamp('2006-07-01')
pre = df[df['date'] < cutoff]
post = df[df['date'] >= cutoff]

print(f"Pre  Jun 2006: {len(pre):,} rows ({pre['date'].min().date()} → {pre['date'].max().date()})")
print(f"Post Jun 2006: {len(post):,} rows ({post['date'].min().date()} → {post['date'].max().date()})")

print(f"\n--- Columns with ANY NaN post June 2006 ---")
print(f"{'Column':<35s} {'Pre NaN%':>9s}  {'Post NaN%':>10s}  {'Post NaN rows':>14s}")
print("-" * 75)
for c in factor_cols:
    pre_nan = pre[c].isna().mean() * 100
    post_nan = post[c].isna().mean() * 100
    post_n = post[c].isna().sum()
    if post_n > 0:
        print(f"{c:<35s} {pre_nan:>8.2f}%  {post_nan:>9.2f}%  {post_n:>14,d}")

# Count columns with zero NaN post-2006
zero_post = sum(1 for c in factor_cols if post[c].isna().sum() == 0)
print(f"\nColumns with ZERO NaN post June 2006: {zero_post} / {len(factor_cols)}")

# ── 2. Largest consecutive gaps per column (post June 2006) ──────────────────
print(f"\n--- Largest consecutive NaN gaps post June 2006 (top 20) ---")
print(f"{'Column':<35s} {'Max Gap (days)':>14s}  {'Gap Start':>12s}  {'Gap End':>12s}")
print("-" * 80)

gap_results = []
for c in factor_cols:
    s = post[['date', c]].copy()
    is_nan = s[c].isna()
    if not is_nan.any():
        continue
    # Find consecutive NaN runs
    groups = (is_nan != is_nan.shift()).cumsum()
    nan_runs = s[is_nan].groupby(groups[is_nan])
    for _, run in nan_runs:
        gap_len = len(run)
        gap_start = run['date'].iloc[0]
        gap_end = run['date'].iloc[-1]
        gap_results.append((c, gap_len, gap_start, gap_end))

gap_results.sort(key=lambda x: -x[1])
for c, gap_len, gs, ge in gap_results[:20]:
    print(f"{c:<35s} {gap_len:>14d}  {gs.date()!s:>12s}  {ge.date()!s:>12s}")

# ── 3. Forward-fill extent from Stage 1 merge ───────────────────────────────
# The Stage 1 merge did ffill(limit=5) on daily sources and merge_asof on weekly.
# merge_asof inherently forward-fills (each weekly value persists until next update).
# Check: how many consecutive same-values exist for weekly sources?

print(f"\n--- Forward-fill extent for weekly sources (how many days each value persists) ---")
weekly_cols = ['initial_claims', 'continued_claims', 'fed_assets', 'tga', 'reserves',
               'bank_credit', 'ci_loans', 'bullish', 'neutral', 'bearish',
               'bullish_8w_ma', 'bull_bear_spread']
weekly_cols = [c for c in weekly_cols if c in post.columns]

for c in weekly_cols:
    s = post[c].dropna()
    # Count how many consecutive days have the same value
    changes = (s != s.shift()).sum()
    total = len(s)
    avg_persist = total / max(changes, 1)
    print(f"  {c:<30s} updates: {changes:>5,d}  avg persistence: {avg_persist:.1f} trading days")

# ── 4. CFTC: how long do positions persist between updates? ──────────────────
print(f"\n--- CFTC update frequency (post June 2006) ---")
cftc_cols = [c for c in factor_cols if c.startswith(('lev_', 'am_', 'dealer_', 'other_', 'open_interest'))]
if cftc_cols:
    c = cftc_cols[0]  # check one representative
    s = post[c].dropna()
    changes = (s != s.shift()).sum()
    total = len(s)
    avg_persist = total / max(changes, 1)
    print(f"  {c}: {changes} updates over {total} days = avg {avg_persist:.1f} day persistence")
    print(f"  (Expected: ~5 trading days between weekly CFTC updates)")

Pre  Jun 2006: 629 rows (2004-01-02 → 2006-06-30)
Post Jun 2006: 4,656 rows (2006-07-03 → 2024-12-31)

--- Columns with ANY NaN post June 2006 ---
Column                               Pre NaN%   Post NaN%   Post NaN rows
---------------------------------------------------------------------------
vxn                                     0.00%       0.02%               1
vxno                                    0.00%       0.02%               1
vxnh                                    0.00%       0.02%               1
vxnl                                    0.00%       0.02%               1
widx_chn                                0.95%       0.11%               5
widx_jpn                                0.16%       0.02%               1
widx_kor                                0.00%       0.02%               1

Columns with ZERO NaN post June 2006: 134 / 141

--- Largest consecutive NaN gaps post June 2006 (top 20) ---
Column                              Max Gap (days)     Gap Start       Gap

In [4]:
# NO DATE TRIM. Panel C keeps 2004-01-02 onward.
#
# The old code trimmed to 2006-07-01 because CFTC starts then and the assert
# below demanded zero NaN. That discarded 2.5 years of history for every OTHER
# factor -- FRED daily, WRDS, VIX and FF5 all start 2004 -- in order to satisfy
# one late-starting source. Stage 3 now runs a PER-FEATURE warm-up beginning in
# each factor's first adequately-covered year, so structural NaN is handled
# where it belongs instead of by deleting rows.

# Fill single-day foreign holiday gaps (unchanged)
sporadic_cols = ['vxn', 'vxno', 'vxnh', 'vxnl', 'widx_chn', 'widx_jpn', 'widx_kor']
sporadic_cols = [c for c in sporadic_cols if c in df.columns]
n_pre = df[sporadic_cols].isna().sum().sum()
df[sporadic_cols] = df[sporadic_cols].ffill(limit=2)
print(f"  Sporadic holiday ffill(limit=2): {n_pre} -> "
      f"{df[sporadic_cols].isna().sum().sum()} NaN")

# ── Report NaN instead of asserting it away ─────────────────────────────────
# Structural NaN is now EXPECTED: CFTC starts Jun 2006, VIX futures Mar 2004,
# twexb/twexm late. These are not errors.
print(f"\n  Rows: {len(df):,}   "
      f"Dates: {df['date'].min().date()} -> {df['date'].max().date()}")

nan_by_col = df[factor_cols].isna().sum()
nan_by_col = nan_by_col[nan_by_col > 0].sort_values(ascending=False)
print(f"  Factors with any NaN: {len(nan_by_col)} / {len(factor_cols)}")
if len(nan_by_col):
    print(f"  {'Column':<32s} {'NaN':>7s} {'%':>7s} {'First valid':>12s}")
    print("  " + "-" * 62)
    for c in nan_by_col.head(15).index:
        fv = df.loc[df[c].notna(), 'date'].min()
        print(f"  {c:<32s} {int(nan_by_col[c]):>7,d} "
              f"{nan_by_col[c] / len(df) * 100:>6.2f}% {fv.date()!s:>12s}")
    if len(nan_by_col) > 15:
        print(f"  ... and {len(nan_by_col) - 15} more")
print(f"\n  These are structural start dates, not errors. Stage 3 reads the")
print(f"  coverage CSV to set each factor's warm-up.")

  Sporadic holiday ffill(limit=2): 18 -> 1 NaN

  Rows: 5,285   Dates: 2004-01-02 -> 2024-12-31
  Factors with any NaN: 50 / 141
  Column                               NaN       %  First valid
  --------------------------------------------------------------
  am_net_chg                           624  11.81%   2006-06-26
  lev_net_chg                          624  11.81%   2006-06-26
  am_spread                            619  11.71%   2006-06-19
  open_interest                        619  11.71%   2006-06-19
  lev_spread                           619  11.71%   2006-06-19
  lev_short                            619  11.71%   2006-06-19
  lev_long                             619  11.71%   2006-06-19
  dealer_long                          619  11.71%   2006-06-19
  dealer_short                         619  11.71%   2006-06-19
  dealer_spread                        619  11.71%   2006-06-19
  other_long                           619  11.71%   2006-06-19
  other_short                         

In [5]:
# %% [markdown]
# ## Block 2: Complete Factor Inventory
#
# Every surviving factor catalogued with source, category, and description.
# Saved as CSV for reference throughout the pipeline.

# %%
print("=" * 90)
print("BLOCK 2: COMPLETE FACTOR INVENTORY")
print("=" * 90)

inventory = []

def add(col, source, category, description):
    inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ─────────────────────────────────────────────────────────────────────────────
# TREASURY YIELDS (11) — FRED daily, constant maturity rates
# ─────────────────────────────────────────────────────────────────────────────

add('yield_1m',  'FRED', 'interest_rates', '1-month Treasury constant maturity yield (%)')
add('yield_3m',  'FRED', 'interest_rates', '3-month Treasury constant maturity yield (%)')
add('yield_6m',  'FRED', 'interest_rates', '6-month Treasury constant maturity yield (%)')
add('yield_1y',  'FRED', 'interest_rates', '1-year Treasury constant maturity yield (%)')
add('yield_2y',  'FRED', 'interest_rates', '2-year Treasury constant maturity yield (%)')
add('yield_3y',  'FRED', 'interest_rates', '3-year Treasury constant maturity yield (%)')
add('yield_5y',  'FRED', 'interest_rates', '5-year Treasury constant maturity yield (%)')
add('yield_7y',  'FRED', 'interest_rates', '7-year Treasury constant maturity yield (%)')
add('yield_10y', 'FRED', 'interest_rates', '10-year Treasury constant maturity yield (%)')
add('yield_20y', 'FRED', 'interest_rates', '20-year Treasury constant maturity yield (%)')
add('yield_30y', 'FRED', 'interest_rates', '30-year Treasury constant maturity yield (%)')

# ─────────────────────────────────────────────────────────────────────────────
# TIPS & BREAKEVENS (6) — FRED daily, inflation expectations
# ─────────────────────────────────────────────────────────────────────────────

add('tips_5y',        'FRED', 'inflation',      '5-year TIPS real yield (%)')
add('tips_7y',        'FRED', 'inflation',      '7-year TIPS real yield (%)')
add('tips_10y',       'FRED', 'inflation',      '10-year TIPS real yield (%)')
add('tips_20y',       'FRED', 'inflation',      '20-year TIPS real yield (%)')
add('breakeven_5y',   'FRED', 'inflation',      '5-year breakeven inflation rate: nominal - TIPS (%)')
add('breakeven_10y',  'FRED', 'inflation',      '10-year breakeven inflation rate: nominal - TIPS (%)')

# ─────────────────────────────────────────────────────────────────────────────
# POLICY RATES (4) — FRED daily + Fama-French
# ─────────────────────────────────────────────────────────────────────────────

add('fed_funds_eff', 'FRED', 'monetary_policy', 'Effective federal funds rate (%)')
add('prime_rate',    'FRED', 'monetary_policy', 'Bank prime loan rate (%)')
add('discount_rate', 'FRED', 'monetary_policy', 'Federal Reserve primary credit discount rate (%)')
add('rf',            'WRDS', 'monetary_policy', 'Daily risk-free rate (1-month T-bill / 252)')

# ─────────────────────────────────────────────────────────────────────────────
# CREDIT SPREADS — OAS (9) — FRED daily, ICE BofA indices
# ─────────────────────────────────────────────────────────────────────────────

add('hy_oas',  'FRED', 'credit', 'High yield OAS: ICE BofA US High Yield index spread (%)')
add('ig_oas',  'FRED', 'credit', 'Investment grade OAS: ICE BofA US Corporate index spread (%)')
add('aaa_oas', 'FRED', 'credit', 'AAA-rated OAS (%)')
add('aa_oas',  'FRED', 'credit', 'AA-rated OAS (%)')
add('a_oas',   'FRED', 'credit', 'A-rated OAS (%)')
add('bbb_oas', 'FRED', 'credit', 'BBB-rated OAS (%)')
add('bb_oas',  'FRED', 'credit', 'BB-rated OAS (%)')
add('b_oas',   'FRED', 'credit', 'B-rated OAS (%)')
add('ccc_oas', 'FRED', 'credit', 'CCC-and-below OAS (%)')

# ─────────────────────────────────────────────────────────────────────────────
# MOODY'S CORPORATE YIELDS (2) — FRED daily
# ─────────────────────────────────────────────────────────────────────────────

add('moody_aaa', 'FRED', 'credit', 'Moody\'s AAA-rated corporate bond yield (%)')
add('moody_baa', 'FRED', 'credit', 'Moody\'s BAA-rated corporate bond yield (%)')

# ─────────────────────────────────────────────────────────────────────────────
# YIELD CURVE & SPREAD DERIVED (13) — computed in FRED cleaning
# ─────────────────────────────────────────────────────────────────────────────

add('slope_2y10y',     'FRED', 'yield_curve', 'Yield curve slope: 10y - 2y (%). Positive = normal, negative = inverted.')
add('slope_3m10y',     'FRED', 'yield_curve', 'Yield curve slope: 10y - 3m (%). Classic recession predictor.')
add('slope_2y30y',     'FRED', 'yield_curve', 'Yield curve slope: 30y - 2y (%)')
add('slope_1y10y',     'FRED', 'yield_curve', 'Yield curve slope: 10y - 1y (%)')
add('curve_2_5_10',    'FRED', 'yield_curve', 'Butterfly: 2×5y - 2y - 10y. Curvature of mid-term segment.')
add('curve_2_10_30',   'FRED', 'yield_curve', 'Butterfly: 2×10y - 2y - 30y. Curvature of long-end.')
add('bbb_aaa_spread',  'FRED', 'credit',      'Credit quality spread: BBB OAS - AAA OAS (%). Risk appetite measure.')
add('bb_bbb_spread',   'FRED', 'credit',      'Crossover spread: BB OAS - BBB OAS (%). Junk-to-IG transition premium.')
add('moody_baa_aaa',   'FRED', 'credit',      'Moody\'s default spread: BAA yield - AAA yield (%)')
add('real_rate_10y',   'FRED', 'inflation',    'Real 10-year rate: 10y TIPS yield (%)')
add('real_rate_5y',    'FRED', 'inflation',    'Real 5-year rate: 5y TIPS yield (%)')
add('ff_2y_spread',    'FRED', 'monetary_policy', 'Fed funds to 2y spread: 2y yield - fed funds (%). Market rate expectations.')
add('ff_10y_spread',   'FRED', 'monetary_policy', 'Fed funds to 10y spread: 10y yield - fed funds (%). Term premium proxy.')

# ─────────────────────────────────────────────────────────────────────────────
# COMMODITIES (4) — FRED daily
# ─────────────────────────────────────────────────────────────────────────────

add('wti_oil',          'FRED', 'commodities', 'WTI crude oil spot price ($/barrel)')
add('brent_oil',        'FRED', 'commodities', 'Brent crude oil spot price ($/barrel)')
add('natgas',           'FRED', 'commodities', 'Henry Hub natural gas spot price ($/mmBtu)')
add('brent_wti_spread', 'FRED', 'commodities', 'Brent-WTI spread ($/barrel). Positive = Brent premium.')

# ─────────────────────────────────────────────────────────────────────────────
# FX TRADE-WEIGHTED INDICES (2) — FRED daily
# ─────────────────────────────────────────────────────────────────────────────

add('twexb', 'FRED', 'fx', 'Trade-weighted US dollar index: broad basket (index level)')
add('twexm', 'FRED', 'fx', 'Trade-weighted US dollar index: major currencies (index level)')

# ─────────────────────────────────────────────────────────────────────────────
# FX RATES (11) — WRDS macro daily, units per USD (except EUR/GBP)
# ─────────────────────────────────────────────────────────────────────────────

add('fx_jpy', 'WRDS', 'fx', 'USD/JPY exchange rate (yen per dollar)')
add('fx_chf', 'WRDS', 'fx', 'USD/CHF exchange rate (francs per dollar)')
add('fx_cad', 'WRDS', 'fx', 'USD/CAD exchange rate (CAD per dollar)')
add('fx_aud', 'WRDS', 'fx', 'USD/AUD exchange rate (AUD per dollar)')
add('fx_nok', 'WRDS', 'fx', 'USD/NOK exchange rate (krone per dollar)')
add('fx_cny', 'WRDS', 'fx', 'USD/CNY exchange rate (yuan per dollar)')
add('fx_krw', 'WRDS', 'fx', 'USD/KRW exchange rate (won per dollar)')
add('fx_brl', 'WRDS', 'fx', 'USD/BRL exchange rate (real per dollar)')
add('fx_mxn', 'WRDS', 'fx', 'USD/MXN exchange rate (peso per dollar)')
add('fx_eur', 'WRDS', 'fx', 'EUR/USD exchange rate (dollars per euro, inverted)')
add('fx_gbp', 'WRDS', 'fx', 'GBP/USD exchange rate (dollars per pound, inverted)')

# ─────────────────────────────────────────────────────────────────────────────
# VIX FAMILY (12) — WRDS macro daily, CBOE volatility indices
# ─────────────────────────────────────────────────────────────────────────────

add('vix',  'WRDS', 'volatility', 'VIX: S&P 500 30-day implied volatility (close)')
add('vixo', 'WRDS', 'volatility', 'VIX open')
add('vixh', 'WRDS', 'volatility', 'VIX high')
add('vixl', 'WRDS', 'volatility', 'VIX low')
add('vxn',  'WRDS', 'volatility', 'VXN: NASDAQ-100 30-day implied volatility (close)')
add('vxno', 'WRDS', 'volatility', 'VXN open')
add('vxnh', 'WRDS', 'volatility', 'VXN high')
add('vxnl', 'WRDS', 'volatility', 'VXN low')
add('vxd',  'WRDS', 'volatility', 'VXD: DJIA 30-day implied volatility (close)')
add('vxdo', 'WRDS', 'volatility', 'VXD open')
add('vxdh', 'WRDS', 'volatility', 'VXD high')
add('vxdl', 'WRDS', 'volatility', 'VXD low')

# ─────────────────────────────────────────────────────────────────────────────
# FAMA-FRENCH FACTORS + MOMENTUM (6) — WRDS macro daily
# ─────────────────────────────────────────────────────────────────────────────

add('mktrf', 'WRDS', 'factor_returns', 'Market excess return: value-weighted CRSP return minus risk-free rate')
add('smb',   'WRDS', 'factor_returns', 'Small minus Big: size factor daily return')
add('hml',   'WRDS', 'factor_returns', 'High minus Low: value factor daily return')
add('rmw',   'WRDS', 'factor_returns', 'Robust minus Weak: profitability factor daily return')
add('cma',   'WRDS', 'factor_returns', 'Conservative minus Aggressive: investment factor daily return')
add('umd',   'WRDS', 'factor_returns', 'Up minus Down: momentum factor daily return')

# ─────────────────────────────────────────────────────────────────────────────
# WORLD INDEX RETURNS (12) — WRDS macro daily
# ─────────────────────────────────────────────────────────────────────────────

add('widx_aus', 'WRDS', 'global_equity', 'Australia equity index daily return')
add('widx_bra', 'WRDS', 'global_equity', 'Brazil equity index daily return')
add('widx_che', 'WRDS', 'global_equity', 'Switzerland equity index daily return')
add('widx_chn', 'WRDS', 'global_equity', 'China equity index daily return')
add('widx_deu', 'WRDS', 'global_equity', 'Germany equity index daily return')
add('widx_fra', 'WRDS', 'global_equity', 'France equity index daily return')
add('widx_gbr', 'WRDS', 'global_equity', 'UK equity index daily return')
add('widx_hkg', 'WRDS', 'global_equity', 'Hong Kong equity index daily return')
add('widx_ind', 'WRDS', 'global_equity', 'India equity index daily return')
add('widx_jpn', 'WRDS', 'global_equity', 'Japan equity index daily return')
add('widx_kor', 'WRDS', 'global_equity', 'South Korea equity index daily return')
add('widx_mex', 'WRDS', 'global_equity', 'Mexico equity index daily return')

# ─────────────────────────────────────────────────────────────────────────────
# VIX FUTURES & TERM STRUCTURE (9) — VIX/SKEW cleaning + derived
# ─────────────────────────────────────────────────────────────────────────────

add('vix_fut_front',        'CBOE', 'vol_term_structure', 'VIX front-month futures settlement price')
add('vix_fut_volume',       'CBOE', 'vol_term_structure', 'VIX front-month futures daily volume')
add('vix_fut_oi',           'CBOE', 'vol_term_structure', 'VIX front-month futures open interest')
add('vix_fut_second',       'CBOE', 'vol_term_structure', 'VIX second-month futures settlement price')
add('vix_term_spread',      'CBOE', 'vol_term_structure', 'VIX term spread: front - second month. Negative = contango (normal).')
add('vix_term_ratio',       'CBOE', 'vol_term_structure', 'VIX term ratio: front / second month. <1 = contango.')
add('vix_fut_ret_1d',       'CBOE', 'vol_term_structure', 'VIX front-month futures 1-day return')
add('vix_term_spread_5d_chg','CBOE','vol_term_structure', '5-day change in VIX term spread (flattening/steepening)')
add('vix_futures_basis',    'Derived','vol_term_structure','VIX futures basis: front futures - VIX spot. Positive = contango.')

# ─────────────────────────────────────────────────────────────────────────────
# CBOE SKEW (6) — VIX/SKEW cleaning + derived
# ─────────────────────────────────────────────────────────────────────────────

add('skew',              'CBOE', 'tail_risk', 'CBOE SKEW index: measures S&P 500 tail risk. 100 = normal, >130 = elevated.')
add('skew_excess',       'CBOE', 'tail_risk', 'SKEW excess: SKEW - 100. How far above normal tail risk.')
add('skew_pctile_252d',  'CBOE', 'tail_risk', 'SKEW percentile rank over trailing 252 days (0-100)')
add('skew_chg_5d',       'CBOE', 'tail_risk', '5-day change in SKEW index')
add('skew_ma20',         'CBOE', 'tail_risk', 'SKEW 20-day moving average (smoothed tail risk)')
add('skew_vs_ma20',      'CBOE', 'tail_risk', 'SKEW deviation from 20-day MA (tail risk surprise)')

# ─────────────────────────────────────────────────────────────────────────────
# CFTC POSITIONING (22) — CFTC weekly, VIX futures positions
# ─────────────────────────────────────────────────────────────────────────────

add('lev_long',      'CFTC', 'positioning', 'Leveraged funds long contracts (VIX futures)')
add('lev_short',     'CFTC', 'positioning', 'Leveraged funds short contracts')
add('lev_spread',    'CFTC', 'positioning', 'Leveraged funds spread (long + short in spreads)')
add('am_long',       'CFTC', 'positioning', 'Asset managers long contracts')
add('am_short',      'CFTC', 'positioning', 'Asset managers short contracts')
add('am_spread',     'CFTC', 'positioning', 'Asset managers spread contracts')
add('dealer_long',   'CFTC', 'positioning', 'Dealer/intermediary long contracts')
add('dealer_short',  'CFTC', 'positioning', 'Dealer/intermediary short contracts')
add('dealer_spread', 'CFTC', 'positioning', 'Dealer/intermediary spread contracts')
add('other_long',    'CFTC', 'positioning', 'Other reportables long contracts')
add('other_short',   'CFTC', 'positioning', 'Other reportables short contracts')
add('other_spread',  'CFTC', 'positioning', 'Other reportables spread contracts')
add('open_interest', 'CFTC', 'positioning', 'Total open interest in VIX futures contracts')
add('lev_net',       'CFTC', 'positioning', 'Leveraged funds net position (long - short)')
add('am_net',        'CFTC', 'positioning', 'Asset managers net position (long - short)')
add('dealer_net',    'CFTC', 'positioning', 'Dealer net position (long - short)')
add('lev_net_pct',   'CFTC', 'positioning', 'Leveraged funds net as % of open interest')
add('am_net_pct',    'CFTC', 'positioning', 'Asset managers net as % of open interest')
add('dealer_net_pct','CFTC', 'positioning', 'Dealer net as % of open interest')
add('lev_am_ratio',  'CFTC', 'positioning', 'Leverage-to-AM ratio: lev_net / am_net. Negative = opposing positions.')
add('lev_net_chg',   'CFTC', 'positioning', 'Weekly change in leveraged funds net position')
add('am_net_chg',    'CFTC', 'positioning', 'Weekly change in asset managers net position')

# ─────────────────────────────────────────────────────────────────────────────
# AAII SENTIMENT (5) — AAII weekly survey
# ─────────────────────────────────────────────────────────────────────────────

add('bullish',          'AAII', 'sentiment', 'AAII % bullish (0-1)')
add('neutral',          'AAII', 'sentiment', 'AAII % neutral (0-1)')
add('bearish',          'AAII', 'sentiment', 'AAII % bearish (0-1)')
add('bullish_8w_ma',    'AAII', 'sentiment', 'AAII bullish 8-week moving average')
add('bull_bear_spread', 'AAII', 'sentiment', 'AAII bull-bear spread: bullish - bearish')

# ─────────────────────────────────────────────────────────────────────────────
# FRED WEEKLY (7) — DoL claims + Fed balance sheet + banking
# ─────────────────────────────────────────────────────────────────────────────

add('initial_claims',   'FRED', 'labor_market',  'Initial jobless claims (persons)')
add('continued_claims', 'FRED', 'labor_market',  'Continued jobless claims (persons)')
add('fed_assets',       'FRED', 'fed_balance',   'Federal Reserve total assets (millions $)')
add('tga',              'FRED', 'fed_balance',   'Treasury General Account balance (millions $)')
add('reserves',         'FRED', 'fed_balance',   'Bank reserves at the Fed (millions $)')
add('bank_credit',      'FRED', 'banking',       'Commercial bank total credit (billions $)')
add('ci_loans',         'FRED', 'banking',       'Commercial & industrial loans (billions $)')

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD AND VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(inventory)

remaining_factors = [c for c in df.columns if c != 'date']
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summaries
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'macro_daily_descriptions_pre.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Saved: {csv_path}")
print(f"    {len(inv)} factors")

# Full table
print(f"\n  Complete inventory:")
print(f"  {'#':<4s} {'Column':<35s} {'Source':<8s} {'Category':<22s} Description")
print("  " + "-" * 110)
for i, row in inv.iterrows():
    print(f"  {i+1:<4d} {row['column']:<35s} {row['source']:<8s} "
          f"{row['category']:<22s} {row['description']}")

BLOCK 2: COMPLETE FACTOR INVENTORY

  Factors in data:       141
  Factors catalogued:    141
  ✓ Every factor in data is catalogued
  ✓ Every catalogued factor exists in data

  By source:
source
FRED       57
WRDS       42
CFTC       22
CBOE       14
AAII        5
Derived     1

  By category:
category
positioning           22
credit                14
fx                    13
global_equity         12
volatility            12
interest_rates        11
vol_term_structure     9
inflation              8
tail_risk              6
factor_returns         6
yield_curve            6
monetary_policy        6
sentiment              5
commodities            4
fed_balance            3
labor_market           2
banking                2

  ✓ Saved: ..\..\..\Data\Data_Collection\Final\Stage_1_5_Validation_and_Feature_Engineering\macro_daily_descriptions_pre.csv
    141 factors

  Complete inventory:
  #    Column                              Source   Category               Description
  ---------------

In [6]:
# %% [markdown]
# ## Block 3: Feature Engineering
#
# Six sections:
#   A. Drop redundancies (exact duplicates, derive VIX intraday then drop O/H/L)
#   B. Daily changes in key levels (1d and 5d for yields, credit, VIX)
#   C. Returns from level data (FX, commodities, dollar index)
#   D. Rolling dynamics (moving averages, factor momentum, volatility-of-vol)
#   E. Regime & cross-asset signals (VIX regime, curve inversion, stock-bond corr)
#   F. Calendar features (using TRADING calendar, not calendar days)
#
# Design principle: keep original levels (z-scoring in Step 3 detrends them)
# AND add explicit change/return features where momentum matters.

# %%
print("=" * 90)
print("BLOCK 3: FEATURE ENGINEERING")
print("=" * 90)

n_before = df.shape[1]
new_features = []

# Ensure sorted by date
df = df.sort_values('date').reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════════
# A. DROP REDUNDANCIES & DERIVE VIX INTRADAY FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- A. Drop redundancies & derive VIX intraday features ---\n")

a_start = len(new_features)

# ── A1. Exact duplicates ────────────────────────────────────────────────────
# real_rate_10y is identical to tips_10y (TIPS yield IS the real rate)
# real_rate_5y is identical to tips_5y
drop_dupes = []
if all(c in df.columns for c in ['real_rate_10y', 'tips_10y']):
    if df['real_rate_10y'].equals(df['tips_10y']):
        drop_dupes.append('real_rate_10y')
        print(f"  Confirmed: real_rate_10y == tips_10y → drop real_rate_10y")
    else:
        print(f"  real_rate_10y ≠ tips_10y — keeping both")

if all(c in df.columns for c in ['real_rate_5y', 'tips_5y']):
    if df['real_rate_5y'].equals(df['tips_5y']):
        drop_dupes.append('real_rate_5y')
        print(f"  Confirmed: real_rate_5y == tips_5y → drop real_rate_5y")
    else:
        print(f"  real_rate_5y ≠ tips_5y — keeping both")

# ── A2. VIX family intraday features (derive then drop O/H/L) ───────────────
for prefix, close_col in [('vix', 'vix'), ('vxn', 'vxn'), ('vxd', 'vxd')]:
    h_col = f'{prefix}h'
    l_col = f'{prefix}l'
    o_col = f'{prefix}o'

    if all(c in df.columns for c in [close_col, h_col, l_col, o_col]):
        # Intraday range: (high - low) / close
        range_name = f'{prefix}_intraday_range'
        df[range_name] = (df[h_col] - df[l_col]) / df[close_col].replace(0, np.nan)
        new_features.append(range_name)

        # Overnight gap: (open - prev close) / prev close
        gap_name = f'{prefix}_overnight_gap'
        df[gap_name] = (df[o_col] - df[close_col].shift(1)) / df[close_col].shift(1).replace(0, np.nan)
        new_features.append(gap_name)

# Drop VIX O/H/L columns (information now captured by range + gap)
vix_ohlc_drop = ['vixo', 'vixh', 'vixl', 'vxno', 'vxnh', 'vxnl', 'vxdo', 'vxdh', 'vxdl']
vix_ohlc_drop = [c for c in vix_ohlc_drop if c in df.columns]

# VIX relative to VXN/VXD (sector volatility spread)
if all(c in df.columns for c in ['vix', 'vxn']):
    df['vix_vxn_ratio'] = df['vix'] / df['vxn'].replace(0, np.nan)
    new_features.append('vix_vxn_ratio')

if all(c in df.columns for c in ['vix', 'vxd']):
    df['vix_vxd_ratio'] = df['vix'] / df['vxd'].replace(0, np.nan)
    new_features.append('vix_vxd_ratio')

# Execute drops
all_drops_a = drop_dupes + vix_ohlc_drop
drops_present = [c for c in all_drops_a if c in df.columns]
df = df.drop(columns=drops_present)

print(f"  Derived: {len(new_features) - a_start} VIX intraday features")
print(f"  Dropped: {len(drops_present)} redundant/replaced columns")
for c in drops_present:
    print(f"    {c}")

# ═══════════════════════════════════════════════════════════════════════════════
# B. DAILY CHANGES IN KEY LEVELS (1d AND 5d)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- B. Daily changes in key levels ---\n")

b_start = len(new_features)

# Yield changes (rates move in basis points — raw diff is appropriate)
yield_change_cols = [
    ('yield_2y',  'yield_2y_chg_1d',  'yield_2y_chg_5d'),
    ('yield_10y', 'yield_10y_chg_1d', 'yield_10y_chg_5d'),
    ('yield_30y', 'yield_30y_chg_1d', 'yield_30y_chg_5d'),
]

for src, name_1d, name_5d in yield_change_cols:
    if src in df.columns:
        df[name_1d] = df[src].diff(1)
        new_features.append(name_1d)
        df[name_5d] = df[src].diff(5)
        new_features.append(name_5d)

# Yield curve slope changes
slope_change_cols = [
    ('slope_2y10y', 'slope_2y10y_chg_1d', 'slope_2y10y_chg_5d'),
    ('slope_3m10y', 'slope_3m10y_chg_1d', 'slope_3m10y_chg_5d'),
]

for src, name_1d, name_5d in slope_change_cols:
    if src in df.columns:
        df[name_1d] = df[src].diff(1)
        new_features.append(name_1d)
        df[name_5d] = df[src].diff(5)
        new_features.append(name_5d)

# Credit spread changes
credit_change_cols = [
    ('hy_oas', 'hy_oas_chg_1d', 'hy_oas_chg_5d'),
    ('ig_oas', 'ig_oas_chg_1d', 'ig_oas_chg_5d'),
    ('bbb_aaa_spread', 'bbb_aaa_chg_1d', 'bbb_aaa_chg_5d'),
]

for src, name_1d, name_5d in credit_change_cols:
    if src in df.columns:
        df[name_1d] = df[src].diff(1)
        new_features.append(name_1d)
        df[name_5d] = df[src].diff(5)
        new_features.append(name_5d)

# VIX changes
if 'vix' in df.columns:
    df['vix_chg_1d'] = df['vix'].diff(1)
    new_features.append('vix_chg_1d')
    df['vix_chg_5d'] = df['vix'].diff(5)
    new_features.append('vix_chg_5d')
    df['vix_pct_chg_1d'] = df['vix'].pct_change(1)
    new_features.append('vix_pct_chg_1d')

# Breakeven inflation changes
if 'breakeven_10y' in df.columns:
    df['breakeven_10y_chg_1d'] = df['breakeven_10y'].diff(1)
    new_features.append('breakeven_10y_chg_1d')
    df['breakeven_10y_chg_5d'] = df['breakeven_10y'].diff(5)
    new_features.append('breakeven_10y_chg_5d')

print(f"  Section B: {len(new_features) - b_start} daily change features")

# ═══════════════════════════════════════════════════════════════════════════════
# C. RETURNS FROM LEVEL DATA (FX, COMMODITIES, DOLLAR INDEX)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- C. Returns from level data ---\n")

c_start = len(new_features)

# FX daily returns (key pairs only — model has 11 FX levels for regime)
fx_return_pairs = ['fx_eur', 'fx_jpy', 'fx_gbp', 'fx_aud', 'fx_cny']
for col in fx_return_pairs:
    if col in df.columns:
        ret_name = f'{col}_ret_1d'
        df[ret_name] = df[col].pct_change(1)
        new_features.append(ret_name)

# Dollar index returns
for col in ['twexb', 'twexm']:
    if col in df.columns:
        ret_name = f'{col}_ret_1d'
        df[ret_name] = df[col].pct_change(1)
        new_features.append(ret_name)

# Commodity returns (1d and 5d)
for col in ['wti_oil', 'brent_oil', 'natgas']:
    if col in df.columns:
        ret_name = f'{col}_ret_1d'
        df[ret_name] = df[col].pct_change(1)
        new_features.append(ret_name)

        ret_5d_name = f'{col}_ret_5d'
        df[ret_5d_name] = df[col].pct_change(5)
        new_features.append(ret_5d_name)

print(f"  Section C: {len(new_features) - c_start} return features")

# ═══════════════════════════════════════════════════════════════════════════════
# D. ROLLING DYNAMICS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- D. Rolling dynamics ---\n")

d_start = len(new_features)

# ── D1. Moving averages for mean-reversion detection ────────────────────────

# VIX relative to 20/50-day MA
if 'vix' in df.columns:
    vix_ma20 = df['vix'].rolling(20, min_periods=10).mean()
    df['vix_vs_ma20'] = (df['vix'] - vix_ma20) / vix_ma20.replace(0, np.nan)
    new_features.append('vix_vs_ma20')

    vix_ma50 = df['vix'].rolling(50, min_periods=25).mean()
    df['vix_vs_ma50'] = (df['vix'] - vix_ma50) / vix_ma50.replace(0, np.nan)
    new_features.append('vix_vs_ma50')

# 10y yield relative to 20-day MA
if 'yield_10y' in df.columns:
    y10_ma20 = df['yield_10y'].rolling(20, min_periods=10).mean()
    df['yield_10y_vs_ma20'] = df['yield_10y'] - y10_ma20
    new_features.append('yield_10y_vs_ma20')

# HY OAS relative to 20-day MA
if 'hy_oas' in df.columns:
    hy_ma20 = df['hy_oas'].rolling(20, min_periods=10).mean()
    df['hy_oas_vs_ma20'] = df['hy_oas'] - hy_ma20
    new_features.append('hy_oas_vs_ma20')

# ── D2. Cumulative factor returns (momentum) ────────────────────────────────

for col in ['mktrf', 'smb', 'hml', 'umd']:
    if col in df.columns:
        df[f'{col}_cum_5d'] = df[col].rolling(5, min_periods=3).sum()
        new_features.append(f'{col}_cum_5d')

        df[f'{col}_cum_20d'] = df[col].rolling(20, min_periods=10).sum()
        new_features.append(f'{col}_cum_20d')

# ── D3. Volatility of key series ────────────────────────────────────────────

# Market return volatility (5d and 20d + ratio)
if 'mktrf' in df.columns:
    df['mktrf_vol_5d'] = df['mktrf'].rolling(5, min_periods=3).std()
    new_features.append('mktrf_vol_5d')

    df['mktrf_vol_20d'] = df['mktrf'].rolling(20, min_periods=10).std()
    new_features.append('mktrf_vol_20d')

    df['mktrf_vol_ratio'] = df['mktrf_vol_5d'] / df['mktrf_vol_20d'].replace(0, np.nan)
    new_features.append('mktrf_vol_ratio')

# Yield volatility (volatility of yield changes)
if 'yield_10y' in df.columns:
    y10_chg = df['yield_10y'].diff(1)
    df['yield_10y_vol_20d'] = y10_chg.rolling(20, min_periods=10).std()
    new_features.append('yield_10y_vol_20d')

# Credit spread volatility
if 'hy_oas' in df.columns:
    hy_chg = df['hy_oas'].diff(1)
    df['hy_oas_vol_20d'] = hy_chg.rolling(20, min_periods=10).std()
    new_features.append('hy_oas_vol_20d')

# ── D4. Global equity momentum ──────────────────────────────────────────────

widx_cols = [c for c in df.columns if c.startswith('widx_')]
if widx_cols:
    # Average world return today (global risk appetite)
    df['widx_avg_ret'] = df[widx_cols].mean(axis=1)
    new_features.append('widx_avg_ret')

    # 5-day cumulative world return
    df['widx_avg_cum_5d'] = df['widx_avg_ret'].rolling(5, min_periods=3).sum()
    new_features.append('widx_avg_cum_5d')

    # Dispersion of world returns (global divergence)
    df['widx_dispersion'] = df[widx_cols].std(axis=1)
    new_features.append('widx_dispersion')

print(f"  Section D: {len(new_features) - d_start} rolling dynamics features")

# ═══════════════════════════════════════════════════════════════════════════════
# E. REGIME & CROSS-ASSET SIGNALS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- E. Regime & cross-asset signals ---\n")

e_start = len(new_features)

# VIX regime indicators
if 'vix' in df.columns:
    df['vix_above_20'] = (df['vix'] > 20).astype(float)
    new_features.append('vix_above_20')

    df['vix_above_30'] = (df['vix'] > 30).astype(float)
    new_features.append('vix_above_30')

# Yield curve inversion indicators
if 'slope_2y10y' in df.columns:
    df['curve_inverted_2y10y'] = (df['slope_2y10y'] < 0).astype(float)
    new_features.append('curve_inverted_2y10y')

if 'slope_3m10y' in df.columns:
    df['curve_inverted_3m10y'] = (df['slope_3m10y'] < 0).astype(float)
    new_features.append('curve_inverted_3m10y')

# Credit stress indicator (HY OAS above 5% = stress)
if 'hy_oas' in df.columns:
    df['credit_stress'] = (df['hy_oas'] > 5).astype(float)
    new_features.append('credit_stress')

# Stock-bond correlation (20-day rolling)
if all(c in df.columns for c in ['mktrf', 'yield_10y']):
    y10_chg = df['yield_10y'].diff(1)
    df['stock_bond_corr_20d'] = df['mktrf'].rolling(20, min_periods=10).corr(y10_chg)
    new_features.append('stock_bond_corr_20d')

# Risk appetite composite
if all(c in df.columns for c in ['mktrf', 'hy_oas', 'vix']):
    mktrf_z = (df['mktrf'] - df['mktrf'].rolling(50, min_periods=25).mean()) / \
              df['mktrf'].rolling(50, min_periods=25).std().replace(0, np.nan)
    hy_chg = -df['hy_oas'].diff(1)  # negative change = tightening = risk-on
    hy_z = (hy_chg - hy_chg.rolling(50, min_periods=25).mean()) / \
           hy_chg.rolling(50, min_periods=25).std().replace(0, np.nan)
    vix_chg = -df['vix'].diff(1)  # negative change = VIX falling = risk-on
    vix_z = (vix_chg - vix_chg.rolling(50, min_periods=25).mean()) / \
            vix_chg.rolling(50, min_periods=25).std().replace(0, np.nan)
    df['risk_appetite'] = (mktrf_z + hy_z + vix_z) / 3
    new_features.append('risk_appetite')

print(f"  Section E: {len(new_features) - e_start} regime/cross-asset features")

# ═══════════════════════════════════════════════════════════════════════════════
# F. CALENDAR FEATURES (using TRADING calendar, not calendar days)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n--- F. Calendar features ---\n")

f_start = len(new_features)

# Day of week (Mon=0, Fri=4) — well-documented Monday/Friday effects
df['day_of_week'] = df['date'].dt.dayofweek.astype(float)
new_features.append('day_of_week')

df['is_monday'] = (df['date'].dt.dayofweek == 0).astype(float)
new_features.append('is_monday')

df['is_friday'] = (df['date'].dt.dayofweek == 4).astype(float)
new_features.append('is_friday')

# Month of year (Jan=1, Dec=12) — January effect, tax-loss selling
df['month_of_year'] = df['date'].dt.month.astype(float)
new_features.append('month_of_year')

# Quarter end (rebalancing, window dressing)
df['is_quarter_end'] = df['date'].dt.is_quarter_end.astype(float)
new_features.append('is_quarter_end')

# Trading days to month end (counts actual trading rows remaining, NOT calendar days)
# On the last trading day before a weekend month-end, this correctly shows 0
df['_ym'] = df['date'].dt.to_period('M')
df['trading_days_to_month_end'] = df.groupby('_ym')['date'].transform(
    lambda x: len(x) - np.arange(1, len(x) + 1)
).astype(float)
new_features.append('trading_days_to_month_end')

# Turn of the month: last 2 + first 2 TRADING days
# Uses actual trading days in the dataframe, not calendar dates
# Institutional rebalancing flows concentrate on these days
last_td = df.groupby('_ym')['date'].transform('max')
first_td = df.groupby('_ym')['date'].transform('min')
df['is_turn_of_month'] = (
    (df['date'] >= last_td - pd.Timedelta(days=3)) |
    (df['date'] <= first_td + pd.Timedelta(days=3))
).astype(float)
new_features.append('is_turn_of_month')

df = df.drop(columns=['_ym'])

# Options expiration week — generate 3rd Fridays mathematically
# This survives Good Friday: when 3rd Friday is a market holiday,
# OPEX moves to Thursday but the full week is still flagged
all_months = pd.date_range(df['date'].min().replace(day=1), df['date'].max(), freq='MS')
third_fridays = [d + pd.Timedelta(days=14 + (4 - d.dayofweek) % 7) for d in all_months]

df['is_opex_week'] = 0.0
for opex in third_fridays:
    mask = (df['date'] >= opex - pd.Timedelta(days=4)) & (df['date'] <= opex)
    df.loc[mask, 'is_opex_week'] = 1.0
new_features.append('is_opex_week')

print(f"  Section F: {len(new_features) - f_start} calendar features")

# Verify calendar features
print(f"\n  Calendar feature verification:")
print(f"    is_monday mean:     {df['is_monday'].mean():.3f} (expect ~0.20)")
print(f"    is_friday mean:     {df['is_friday'].mean():.3f} (expect ~0.20)")
print(f"    is_opex_week mean:  {df['is_opex_week'].mean():.3f} (expect ~0.20)")
print(f"    is_turn_of_month:   {df['is_turn_of_month'].mean():.3f} (expect ~0.18-0.25)")
print(f"    trading_days_to_month_end range: "
      f"[{df['trading_days_to_month_end'].min():.0f}, {df['trading_days_to_month_end'].max():.0f}]"
      f" (expect [0, ~22])")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("BLOCK 3: SUMMARY")
print("=" * 90)

remaining = [c for c in df.columns if c != 'date']

sect_a = sum(1 for f in new_features if new_features.index(f) < b_start)
sect_b = sum(1 for f in new_features if b_start <= new_features.index(f) < c_start)
sect_c = sum(1 for f in new_features if c_start <= new_features.index(f) < d_start)
sect_d = sum(1 for f in new_features if d_start <= new_features.index(f) < e_start)
sect_e = sum(1 for f in new_features if e_start <= new_features.index(f) < f_start)
sect_f = sum(1 for f in new_features if new_features.index(f) >= f_start)

print(f"\n  New features by section:")
print(f"    A. VIX intraday + vol ratios:     {sect_a}")
print(f"    B. Daily changes (1d/5d):         {sect_b}")
print(f"    C. Returns from levels:           {sect_c}")
print(f"    D. Rolling dynamics:              {sect_d}")
print(f"    E. Regime/cross-asset:            {sect_e}")
print(f"    F. Calendar features:             {sect_f}")
print(f"    ──────────────────────────────")
print(f"    Total new features:               {len(new_features)}")
print(f"\n  Redundant/replaced dropped:         {len(drops_present)}")
print(f"  Net column change:                  {n_before} → {df.shape[1]} columns")
print(f"  Surviving factors:                  {len(remaining)}")

# Verify date intact
assert 'date' in df.columns
print(f"\n  ✓ Date column intact")

# Verify all new features present
missing_new = [f for f in new_features if f not in df.columns]
if missing_new:
    print(f"  ✗ Missing: {missing_new}")
else:
    print(f"  ✓ All {len(new_features)} new features present")

# NaN check
total_nan = df[remaining].isna().sum().sum()
warmup_rows = 50
warmup_nan = df.head(warmup_rows)[remaining].isna().sum().sum()
post_warmup_nan = df.iloc[warmup_rows:][remaining].isna().sum().sum()
print(f"\n  Total NaN: {total_nan:,}")
print(f"    In first {warmup_rows} rows (warmup): {warmup_nan:,}")
print(f"    After warmup: {post_warmup_nan:,}")

if post_warmup_nan == 0:
    print(f"  ✓ Zero NaN after warmup period")
else:
    post_warmup = df.iloc[warmup_rows:]
    nan_cols = post_warmup[remaining].isna().sum()
    nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
    print(f"  Columns with NaN after row {warmup_rows} "
          f"(expected: CFTC to Jun 2006, VIX futures to Mar 2004, twex):")
    for c in nan_cols.head(10).index:
        print(f"    {c}: {int(nan_cols[c])} NaN")

# Complete feature list
print(f"\n  Complete list of {len(new_features)} new features:")
for i, f in enumerate(new_features, 1):
    print(f"    {i:>3d}. {f}")

BLOCK 3: FEATURE ENGINEERING

--- A. Drop redundancies & derive VIX intraday features ---

  real_rate_10y ≠ tips_10y — keeping both
  real_rate_5y ≠ tips_5y — keeping both
  Derived: 8 VIX intraday features
  Dropped: 9 redundant/replaced columns
    vixo
    vixh
    vixl
    vxno
    vxnh
    vxnl
    vxdo
    vxdh
    vxdl

--- B. Daily changes in key levels ---

  Section B: 21 daily change features

--- C. Returns from level data ---

  Section C: 13 return features

--- D. Rolling dynamics ---

  Section D: 20 rolling dynamics features

--- E. Regime & cross-asset signals ---

  Section E: 7 regime/cross-asset features

--- F. Calendar features ---

  Section F: 8 calendar features

  Calendar feature verification:
    is_monday mean:     0.187 (expect ~0.20)
    is_friday mean:     0.201 (expect ~0.20)
    is_opex_week mean:  0.234 (expect ~0.20)
    is_turn_of_month:   0.305 (expect ~0.18-0.25)
    trading_days_to_month_end range: [0, 22] (expect [0, ~22])

BLOCK 3: SUMMARY

 

In [7]:
# ── Fix: is_turn_of_month using exact trading day rank ───────────────────────
df['_ym'] = df['date'].dt.to_period('M')
df['_rank_from_start'] = df.groupby('_ym').cumcount()
df['_rank_from_end'] = df.groupby('_ym')['date'].transform('count') - df.groupby('_ym').cumcount() - 1

old_mean = df['is_turn_of_month'].mean()
df['is_turn_of_month'] = (
    (df['_rank_from_end'] <= 1) | (df['_rank_from_start'] <= 1)
).astype(float)
new_mean = df['is_turn_of_month'].mean()

df = df.drop(columns=['_rank_from_start', '_rank_from_end', '_ym'])

print(f"is_turn_of_month: {old_mean:.3f} → {new_mean:.3f} (expect ~0.19)")

is_turn_of_month: 0.305 → 0.191 (expect ~0.19)


In [8]:
# %% [markdown]
# ## Block 4: Final Factor Inventory & Save
#
# Catalogues every surviving factor, saves inventory CSV and engineered parquet.

# %%
print("=" * 90)
print("BLOCK 4: FINAL FACTOR INVENTORY & SAVE")
print("=" * 90)

from pathlib import Path

OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD FINAL INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
final_inventory = []

def add(col, source, category, description):
    final_inventory.append({
        'column': col,
        'source': source,
        'category': category,
        'description': description,
    })

# ─────────────────────────────────────────────────────────────────────────────
# TREASURY YIELDS (11) — kept as-is
# ─────────────────────────────────────────────────────────────────────────────

add('yield_1m',  'FRED', 'interest_rates', '1-month Treasury yield (%)')
add('yield_3m',  'FRED', 'interest_rates', '3-month Treasury yield (%)')
add('yield_6m',  'FRED', 'interest_rates', '6-month Treasury yield (%)')
add('yield_1y',  'FRED', 'interest_rates', '1-year Treasury yield (%)')
add('yield_2y',  'FRED', 'interest_rates', '2-year Treasury yield (%)')
add('yield_3y',  'FRED', 'interest_rates', '3-year Treasury yield (%)')
add('yield_5y',  'FRED', 'interest_rates', '5-year Treasury yield (%)')
add('yield_7y',  'FRED', 'interest_rates', '7-year Treasury yield (%)')
add('yield_10y', 'FRED', 'interest_rates', '10-year Treasury yield (%)')
add('yield_20y', 'FRED', 'interest_rates', '20-year Treasury yield (%)')
add('yield_30y', 'FRED', 'interest_rates', '30-year Treasury yield (%)')

# ─────────────────────────────────────────────────────────────────────────────
# TIPS & BREAKEVENS (6)
# ─────────────────────────────────────────────────────────────────────────────

add('tips_5y',       'FRED', 'inflation', '5-year TIPS real yield (%)')
add('tips_7y',       'FRED', 'inflation', '7-year TIPS real yield (%)')
add('tips_10y',      'FRED', 'inflation', '10-year TIPS real yield (%)')
add('tips_20y',      'FRED', 'inflation', '20-year TIPS real yield (%)')
add('breakeven_5y',  'FRED', 'inflation', '5-year breakeven inflation (%)')
add('breakeven_10y', 'FRED', 'inflation', '10-year breakeven inflation (%)')

# ─────────────────────────────────────────────────────────────────────────────
# POLICY RATES (4)
# ─────────────────────────────────────────────────────────────────────────────

add('fed_funds_eff', 'FRED', 'monetary_policy', 'Effective federal funds rate (%)')
add('prime_rate',    'FRED', 'monetary_policy', 'Bank prime loan rate (%)')
add('discount_rate', 'FRED', 'monetary_policy', 'Fed primary credit discount rate (%)')
add('rf',            'WRDS', 'monetary_policy', 'Daily risk-free rate (T-bill / 252)')

# ─────────────────────────────────────────────────────────────────────────────
# CREDIT SPREADS OAS (9)
# ─────────────────────────────────────────────────────────────────────────────

add('hy_oas',  'FRED', 'credit', 'High yield OAS (%)')
add('ig_oas',  'FRED', 'credit', 'Investment grade OAS (%)')
add('aaa_oas', 'FRED', 'credit', 'AAA OAS (%)')
add('aa_oas',  'FRED', 'credit', 'AA OAS (%)')
add('a_oas',   'FRED', 'credit', 'A OAS (%)')
add('bbb_oas', 'FRED', 'credit', 'BBB OAS (%)')
add('bb_oas',  'FRED', 'credit', 'BB OAS (%)')
add('b_oas',   'FRED', 'credit', 'B OAS (%)')
add('ccc_oas', 'FRED', 'credit', 'CCC-and-below OAS (%)')

# ─────────────────────────────────────────────────────────────────────────────
# MOODY'S (2)
# ─────────────────────────────────────────────────────────────────────────────

add('moody_aaa', 'FRED', 'credit', 'Moody\'s AAA corporate yield (%)')
add('moody_baa', 'FRED', 'credit', 'Moody\'s BAA corporate yield (%)')

# ─────────────────────────────────────────────────────────────────────────────
# YIELD CURVE DERIVED (13) — kept as-is
# ─────────────────────────────────────────────────────────────────────────────

add('slope_2y10y',     'FRED', 'yield_curve', 'Yield curve slope: 10y - 2y (%)')
add('slope_3m10y',     'FRED', 'yield_curve', 'Yield curve slope: 10y - 3m (%)')
add('slope_2y30y',     'FRED', 'yield_curve', 'Yield curve slope: 30y - 2y (%)')
add('slope_1y10y',     'FRED', 'yield_curve', 'Yield curve slope: 10y - 1y (%)')
add('curve_2_5_10',    'FRED', 'yield_curve', 'Butterfly: 2×5y - 2y - 10y')
add('curve_2_10_30',   'FRED', 'yield_curve', 'Butterfly: 2×10y - 2y - 30y')
add('bbb_aaa_spread',  'FRED', 'credit',      'Credit quality spread: BBB - AAA OAS (%)')
add('bb_bbb_spread',   'FRED', 'credit',      'Crossover spread: BB - BBB OAS (%)')
add('moody_baa_aaa',   'FRED', 'credit',      'Moody\'s default spread: BAA - AAA (%)')
add('ff_2y_spread',    'FRED', 'monetary_policy', 'Fed funds to 2y spread (%)')
add('ff_10y_spread',   'FRED', 'monetary_policy', 'Fed funds to 10y spread (%)')

# ─────────────────────────────────────────────────────────────────────────────
# COMMODITIES (4)
# ─────────────────────────────────────────────────────────────────────────────

add('wti_oil',          'FRED', 'commodities', 'WTI crude oil spot ($/bbl)')
add('brent_oil',        'FRED', 'commodities', 'Brent crude oil spot ($/bbl)')
add('natgas',           'FRED', 'commodities', 'Henry Hub natural gas ($/mmBtu)')
add('brent_wti_spread', 'FRED', 'commodities', 'Brent-WTI spread ($/bbl)')

# ─────────────────────────────────────────────────────────────────────────────
# FX (13) — 2 trade-weighted + 11 rates, levels kept as-is
# ─────────────────────────────────────────────────────────────────────────────

add('twexb',  'FRED', 'fx', 'Trade-weighted USD: broad basket (index)')
add('twexm',  'FRED', 'fx', 'Trade-weighted USD: major currencies (index)')
add('fx_jpy', 'WRDS', 'fx', 'USD/JPY (yen per dollar)')
add('fx_chf', 'WRDS', 'fx', 'USD/CHF (francs per dollar)')
add('fx_cad', 'WRDS', 'fx', 'USD/CAD')
add('fx_aud', 'WRDS', 'fx', 'USD/AUD')
add('fx_nok', 'WRDS', 'fx', 'USD/NOK')
add('fx_cny', 'WRDS', 'fx', 'USD/CNY')
add('fx_krw', 'WRDS', 'fx', 'USD/KRW')
add('fx_brl', 'WRDS', 'fx', 'USD/BRL')
add('fx_mxn', 'WRDS', 'fx', 'USD/MXN')
add('fx_eur', 'WRDS', 'fx', 'EUR/USD (dollars per euro)')
add('fx_gbp', 'WRDS', 'fx', 'GBP/USD (dollars per pound)')

# ─────────────────────────────────────────────────────────────────────────────
# VIX FAMILY (3) — closes kept, O/H/L dropped after deriving range/gap
# ─────────────────────────────────────────────────────────────────────────────

add('vix', 'WRDS', 'volatility', 'VIX: S&P 500 30d implied vol (close)')
add('vxn', 'WRDS', 'volatility', 'VXN: NASDAQ-100 30d implied vol (close)')
add('vxd', 'WRDS', 'volatility', 'VXD: DJIA 30d implied vol (close)')

# ─────────────────────────────────────────────────────────────────────────────
# FAMA-FRENCH + MOMENTUM (6)
# ─────────────────────────────────────────────────────────────────────────────

add('mktrf', 'WRDS', 'factor_returns', 'Market excess return')
add('smb',   'WRDS', 'factor_returns', 'Small minus Big')
add('hml',   'WRDS', 'factor_returns', 'High minus Low')
add('rmw',   'WRDS', 'factor_returns', 'Robust minus Weak')
add('cma',   'WRDS', 'factor_returns', 'Conservative minus Aggressive')
add('umd',   'WRDS', 'factor_returns', 'Up minus Down (momentum)')

# ─────────────────────────────────────────────────────────────────────────────
# WORLD INDEX RETURNS (12)
# ─────────────────────────────────────────────────────────────────────────────

add('widx_aus', 'WRDS', 'global_equity', 'Australia index daily return')
add('widx_bra', 'WRDS', 'global_equity', 'Brazil index daily return')
add('widx_che', 'WRDS', 'global_equity', 'Switzerland index daily return')
add('widx_chn', 'WRDS', 'global_equity', 'China index daily return')
add('widx_deu', 'WRDS', 'global_equity', 'Germany index daily return')
add('widx_fra', 'WRDS', 'global_equity', 'France index daily return')
add('widx_gbr', 'WRDS', 'global_equity', 'UK index daily return')
add('widx_hkg', 'WRDS', 'global_equity', 'Hong Kong index daily return')
add('widx_ind', 'WRDS', 'global_equity', 'India index daily return')
add('widx_jpn', 'WRDS', 'global_equity', 'Japan index daily return')
add('widx_kor', 'WRDS', 'global_equity', 'South Korea index daily return')
add('widx_mex', 'WRDS', 'global_equity', 'Mexico index daily return')

# ─────────────────────────────────────────────────────────────────────────────
# VIX FUTURES & TERM STRUCTURE (9)
# ─────────────────────────────────────────────────────────────────────────────

add('vix_fut_front',         'CBOE', 'vol_term_structure', 'VIX front-month futures price')
add('vix_fut_volume',        'CBOE', 'vol_term_structure', 'VIX front-month futures volume')
add('vix_fut_oi',            'CBOE', 'vol_term_structure', 'VIX front-month futures open interest')
add('vix_fut_second',        'CBOE', 'vol_term_structure', 'VIX second-month futures price')
add('vix_term_spread',       'CBOE', 'vol_term_structure', 'VIX term spread: front - second')
add('vix_term_ratio',        'CBOE', 'vol_term_structure', 'VIX term ratio: front / second')
add('vix_fut_ret_1d',        'CBOE', 'vol_term_structure', 'VIX front futures 1d return')
add('vix_term_spread_5d_chg','CBOE', 'vol_term_structure', '5d change in VIX term spread')
add('vix_futures_basis',     'Derived','vol_term_structure','VIX futures basis: front - spot')

# ─────────────────────────────────────────────────────────────────────────────
# SKEW (6)
# ─────────────────────────────────────────────────────────────────────────────

add('skew',             'CBOE', 'tail_risk', 'CBOE SKEW index')
add('skew_excess',      'CBOE', 'tail_risk', 'SKEW - 100')
add('skew_pctile_252d', 'CBOE', 'tail_risk', 'SKEW percentile rank 252d (0-100)')
add('skew_chg_5d',      'CBOE', 'tail_risk', '5d change in SKEW')
add('skew_ma20',        'CBOE', 'tail_risk', 'SKEW 20d moving average')
add('skew_vs_ma20',     'CBOE', 'tail_risk', 'SKEW deviation from 20d MA')

# ─────────────────────────────────────────────────────────────────────────────
# CFTC POSITIONING (22)
# ─────────────────────────────────────────────────────────────────────────────

add('lev_long',       'CFTC', 'positioning', 'Leveraged funds long contracts')
add('lev_short',      'CFTC', 'positioning', 'Leveraged funds short contracts')
add('lev_spread',     'CFTC', 'positioning', 'Leveraged funds spread')
add('am_long',        'CFTC', 'positioning', 'Asset managers long contracts')
add('am_short',       'CFTC', 'positioning', 'Asset managers short contracts')
add('am_spread',      'CFTC', 'positioning', 'Asset managers spread')
add('dealer_long',    'CFTC', 'positioning', 'Dealer long contracts')
add('dealer_short',   'CFTC', 'positioning', 'Dealer short contracts')
add('dealer_spread',  'CFTC', 'positioning', 'Dealer spread')
add('other_long',     'CFTC', 'positioning', 'Other reportables long')
add('other_short',    'CFTC', 'positioning', 'Other reportables short')
add('other_spread',   'CFTC', 'positioning', 'Other reportables spread')
add('open_interest',  'CFTC', 'positioning', 'Total VIX futures open interest')
add('lev_net',        'CFTC', 'positioning', 'Leveraged funds net (long - short)')
add('am_net',         'CFTC', 'positioning', 'Asset managers net')
add('dealer_net',     'CFTC', 'positioning', 'Dealer net')
add('lev_net_pct',    'CFTC', 'positioning', 'Leveraged net as % of OI')
add('am_net_pct',     'CFTC', 'positioning', 'Asset managers net as % of OI')
add('dealer_net_pct', 'CFTC', 'positioning', 'Dealer net as % of OI')
add('lev_am_ratio',   'CFTC', 'positioning', 'Leverage-to-AM net ratio')
add('lev_net_chg',    'CFTC', 'positioning', 'Weekly change in leveraged net')
add('am_net_chg',     'CFTC', 'positioning', 'Weekly change in AM net')

# ─────────────────────────────────────────────────────────────────────────────
# AAII SENTIMENT (5)
# ─────────────────────────────────────────────────────────────────────────────

add('bullish',          'AAII', 'sentiment', 'AAII % bullish')
add('neutral',          'AAII', 'sentiment', 'AAII % neutral')
add('bearish',          'AAII', 'sentiment', 'AAII % bearish')
add('bullish_8w_ma',    'AAII', 'sentiment', 'AAII bullish 8-week MA')
add('bull_bear_spread', 'AAII', 'sentiment', 'AAII bull-bear spread')

# ─────────────────────────────────────────────────────────────────────────────
# FRED WEEKLY (7)
# ─────────────────────────────────────────────────────────────────────────────

add('initial_claims',   'FRED', 'labor_market', 'Initial jobless claims')
add('continued_claims', 'FRED', 'labor_market', 'Continued jobless claims')
add('fed_assets',       'FRED', 'fed_balance',  'Fed total assets ($M)')
add('tga',              'FRED', 'fed_balance',  'Treasury General Account ($M)')
add('reserves',         'FRED', 'fed_balance',  'Bank reserves at Fed ($M)')
add('bank_credit',      'FRED', 'banking',      'Commercial bank total credit ($B)')
add('ci_loans',         'FRED', 'banking',      'C&I loans ($B)')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 NEW: VIX INTRADAY FEATURES (Section A)
# ═══════════════════════════════════════════════════════════════════════════════

add('vix_intraday_range',  'Derived', 'volatility',       'VIX (high - low) / close')
add('vix_overnight_gap',   'Derived', 'volatility',       'VIX (open - prev close) / prev close')
add('vxn_intraday_range',  'Derived', 'volatility',       'VXN (high - low) / close')
add('vxn_overnight_gap',   'Derived', 'volatility',       'VXN overnight gap')
add('vxd_intraday_range',  'Derived', 'volatility',       'VXD (high - low) / close')
add('vxd_overnight_gap',   'Derived', 'volatility',       'VXD overnight gap')
add('vix_vxn_ratio',       'Derived', 'volatility',       'VIX / VXN: S&P vol relative to NASDAQ vol')
add('vix_vxd_ratio',       'Derived', 'volatility',       'VIX / VXD: S&P vol relative to DJIA vol')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 NEW: DAILY CHANGES (Section B)
# ═══════════════════════════════════════════════════════════════════════════════

add('yield_2y_chg_1d',      'Derived', 'rate_dynamics',   '1d change in 2y yield')
add('yield_2y_chg_5d',      'Derived', 'rate_dynamics',   '5d change in 2y yield')
add('yield_10y_chg_1d',     'Derived', 'rate_dynamics',   '1d change in 10y yield')
add('yield_10y_chg_5d',     'Derived', 'rate_dynamics',   '5d change in 10y yield')
add('yield_30y_chg_1d',     'Derived', 'rate_dynamics',   '1d change in 30y yield')
add('yield_30y_chg_5d',     'Derived', 'rate_dynamics',   '5d change in 30y yield')
add('slope_2y10y_chg_1d',   'Derived', 'rate_dynamics',   '1d change in 2y10y slope')
add('slope_2y10y_chg_5d',   'Derived', 'rate_dynamics',   '5d change in 2y10y slope')
add('slope_3m10y_chg_1d',   'Derived', 'rate_dynamics',   '1d change in 3m10y slope')
add('slope_3m10y_chg_5d',   'Derived', 'rate_dynamics',   '5d change in 3m10y slope')
add('hy_oas_chg_1d',        'Derived', 'credit_dynamics', '1d change in HY OAS')
add('hy_oas_chg_5d',        'Derived', 'credit_dynamics', '5d change in HY OAS')
add('ig_oas_chg_1d',        'Derived', 'credit_dynamics', '1d change in IG OAS')
add('ig_oas_chg_5d',        'Derived', 'credit_dynamics', '5d change in IG OAS')
add('bbb_aaa_chg_1d',       'Derived', 'credit_dynamics', '1d change in BBB-AAA spread')
add('bbb_aaa_chg_5d',       'Derived', 'credit_dynamics', '5d change in BBB-AAA spread')
add('vix_chg_1d',           'Derived', 'vol_dynamics',    '1d change in VIX (absolute)')
add('vix_chg_5d',           'Derived', 'vol_dynamics',    '5d change in VIX')
add('vix_pct_chg_1d',       'Derived', 'vol_dynamics',    '1d VIX % change')
add('breakeven_10y_chg_1d', 'Derived', 'inflation_dynamics', '1d change in 10y breakeven')
add('breakeven_10y_chg_5d', 'Derived', 'inflation_dynamics', '5d change in 10y breakeven')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 NEW: RETURNS FROM LEVELS (Section C)
# ═══════════════════════════════════════════════════════════════════════════════

add('fx_eur_ret_1d',   'Derived', 'fx_dynamics', 'EUR/USD 1d return')
add('fx_jpy_ret_1d',   'Derived', 'fx_dynamics', 'USD/JPY 1d return')
add('fx_gbp_ret_1d',   'Derived', 'fx_dynamics', 'GBP/USD 1d return')
add('fx_aud_ret_1d',   'Derived', 'fx_dynamics', 'USD/AUD 1d return')
add('fx_cny_ret_1d',   'Derived', 'fx_dynamics', 'USD/CNY 1d return')
add('twexb_ret_1d',    'Derived', 'fx_dynamics', 'Broad dollar index 1d return')
add('twexm_ret_1d',    'Derived', 'fx_dynamics', 'Major dollar index 1d return')
add('wti_oil_ret_1d',  'Derived', 'commodity_dynamics', 'WTI oil 1d return')
add('wti_oil_ret_5d',  'Derived', 'commodity_dynamics', 'WTI oil 5d return')
add('brent_oil_ret_1d','Derived', 'commodity_dynamics', 'Brent oil 1d return')
add('brent_oil_ret_5d','Derived', 'commodity_dynamics', 'Brent oil 5d return')
add('natgas_ret_1d',   'Derived', 'commodity_dynamics', 'Natural gas 1d return')
add('natgas_ret_5d',   'Derived', 'commodity_dynamics', 'Natural gas 5d return')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 NEW: ROLLING DYNAMICS (Section D)
# ═══════════════════════════════════════════════════════════════════════════════

add('vix_vs_ma20',        'Derived', 'vol_dynamics',       'VIX vs 20d MA (% deviation)')
add('vix_vs_ma50',        'Derived', 'vol_dynamics',       'VIX vs 50d MA (% deviation)')
add('yield_10y_vs_ma20',  'Derived', 'rate_dynamics',      '10y yield vs 20d MA (bp deviation)')
add('hy_oas_vs_ma20',     'Derived', 'credit_dynamics',    'HY OAS vs 20d MA (bp deviation)')
add('mktrf_cum_5d',       'Derived', 'factor_momentum',    'Market excess return cumulative 5d')
add('mktrf_cum_20d',      'Derived', 'factor_momentum',    'Market excess return cumulative 20d')
add('smb_cum_5d',         'Derived', 'factor_momentum',    'SMB cumulative 5d')
add('smb_cum_20d',        'Derived', 'factor_momentum',    'SMB cumulative 20d')
add('hml_cum_5d',         'Derived', 'factor_momentum',    'HML cumulative 5d')
add('hml_cum_20d',        'Derived', 'factor_momentum',    'HML cumulative 20d')
add('umd_cum_5d',         'Derived', 'factor_momentum',    'UMD cumulative 5d')
add('umd_cum_20d',        'Derived', 'factor_momentum',    'UMD cumulative 20d')
add('mktrf_vol_5d',       'Derived', 'realised_vol',       'Market return 5d rolling std')
add('mktrf_vol_20d',      'Derived', 'realised_vol',       'Market return 20d rolling std')
add('mktrf_vol_ratio',    'Derived', 'realised_vol',       'Market vol ratio: 5d / 20d')
add('yield_10y_vol_20d',  'Derived', 'rate_dynamics',      '10y yield change 20d rolling std')
add('hy_oas_vol_20d',     'Derived', 'credit_dynamics',    'HY OAS change 20d rolling std')
add('widx_avg_ret',       'Derived', 'global_equity_dynamics', 'Average world index return today')
add('widx_avg_cum_5d',    'Derived', 'global_equity_dynamics', 'Average world index cumulative 5d')
add('widx_dispersion',    'Derived', 'global_equity_dynamics', 'Cross-country return dispersion (std)')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 NEW: REGIME & CROSS-ASSET (Section E)
# ═══════════════════════════════════════════════════════════════════════════════

add('vix_above_20',         'Derived', 'regime', 'VIX > 20 indicator (elevated fear)')
add('vix_above_30',         'Derived', 'regime', 'VIX > 30 indicator (crisis)')
add('curve_inverted_2y10y', 'Derived', 'regime', '2y10y yield curve inverted indicator')
add('curve_inverted_3m10y', 'Derived', 'regime', '3m10y yield curve inverted indicator')
add('credit_stress',        'Derived', 'regime', 'HY OAS > 5% indicator (credit stress)')
add('stock_bond_corr_20d',  'Derived', 'cross_asset', '20d rolling corr(market return, yield change)')
add('risk_appetite',        'Derived', 'cross_asset', 'Composite: z(mktrf) + z(-Δhy_oas) + z(-Δvix)')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCK 3 NEW: CALENDAR (Section F)
# ═══════════════════════════════════════════════════════════════════════════════

add('day_of_week',               'Derived', 'calendar', 'Day of week: Mon=0, Fri=4')
add('is_monday',                 'Derived', 'calendar', 'Monday indicator')
add('is_friday',                 'Derived', 'calendar', 'Friday indicator')
add('month_of_year',             'Derived', 'calendar', 'Month: Jan=1, Dec=12')
add('is_quarter_end',            'Derived', 'calendar', 'Quarter-end day indicator')
add('trading_days_to_month_end', 'Derived', 'calendar', 'Trading days remaining in month')
add('is_turn_of_month',          'Derived', 'calendar', 'Last 2 + first 2 trading days of month')
add('is_opex_week',              'Derived', 'calendar', 'Options expiration week (3rd Friday)')

# ═══════════════════════════════════════════════════════════════════════════════
# VALIDATE INVENTORY VS ACTUAL DATA
# ═══════════════════════════════════════════════════════════════════════════════

# %%
inv = pd.DataFrame(final_inventory)

remaining_factors = [c for c in df.columns if c != 'date']
catalogued = set(inv['column'])

in_data_not_catalogued = [c for c in remaining_factors if c not in catalogued]
in_catalogue_not_data = [c for c in catalogued if c not in remaining_factors]

print(f"\n  Factors in data:       {len(remaining_factors)}")
print(f"  Factors catalogued:    {len(inv)}")

if in_data_not_catalogued:
    print(f"\n  ✗ IN DATA but NOT catalogued ({len(in_data_not_catalogued)}):")
    for c in in_data_not_catalogued:
        print(f"    {c}")
else:
    print(f"  ✓ Every factor in data is catalogued")

if in_catalogue_not_data:
    print(f"\n  ✗ In catalogue but NOT in data ({len(in_catalogue_not_data)}):")
    for c in in_catalogue_not_data:
        print(f"    {c}")
else:
    print(f"  ✓ Every catalogued factor exists in data")

# Summaries
print(f"\n  By source:")
print(inv['source'].value_counts().to_string())

print(f"\n  By category:")
print(inv['category'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE INVENTORY CSV
# ═══════════════════════════════════════════════════════════════════════════════

# %%
csv_path = OUT_DIR / 'macro_daily_factor_inventory_final.csv'
csv_string = inv.to_csv(index=False)
with open(csv_path, 'w', encoding='utf-8') as f:
    f.write(csv_string)
print(f"\n  ✓ Inventory saved: {csv_path}")
print(f"    {len(inv)} factors")

# ═══════════════════════════════════════════════════════════════════════════════
# SAVE ENGINEERED PANEL TO PARQUET
# ═══════════════════════════════════════════════════════════════════════════════

# %%
df = df.sort_values('date').reset_index(drop=True)

parquet_path = OUT_DIR / 'panel_macro_daily_engineered.parquet'
df.to_parquet(parquet_path, index=False, engine='pyarrow')

file_size = parquet_path.stat().st_size
print(f"\n  ✓ Panel saved: {parquet_path}")
print(f"    {len(df):,} rows × {df.shape[1]} columns")
print(f"    Key: date")
print(f"    Factors: {len(remaining_factors)}")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("PANEL C FEATURE ENGINEERING COMPLETE")
print("=" * 90)

print(f"""
  Pipeline: Raw (142 cols) → Block 1 (142) → Block 3 ({df.shape[1]}) → Final ({df.shape[1]})

  Final panel:
    Rows:       {len(df):,}
    Columns:    {df.shape[1]}
    Factors:    {len(remaining_factors)}
    Date range: {df['date'].min().date()} → {df['date'].max().date()}

  Saved to:
    Panel:     {parquet_path}
    Inventory: {csv_path}

  Next step: Panel C is market-level — no cross-sectional aggregation needed.
  These factors merge directly into the model-ready tables in Step 3
  after expanding-window z-standardisation.
""")

BLOCK 4: FINAL FACTOR INVENTORY & SAVE

  Factors in data:       209
  Factors catalogued:    207

  ✗ IN DATA but NOT catalogued (2):
    real_rate_10y
    real_rate_5y
  ✓ Every catalogued factor exists in data

  By source:
source
Derived    78
FRED       55
WRDS       33
CFTC       22
CBOE       14
AAII        5

  By category:
category
positioning               22
credit                    14
fx                        13
rate_dynamics             12
global_equity             12
interest_rates            11
volatility                11
vol_term_structure         9
factor_momentum            8
credit_dynamics            8
calendar                   8
fx_dynamics                7
tail_risk                  6
inflation                  6
yield_curve                6
factor_returns             6
commodity_dynamics         6
monetary_policy            6
regime                     5
sentiment                  5
vol_dynamics               5
commodities                4
fed_balance        